In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip install -q fastapi uvicorn pyngrok transformers torchaudio onnxruntime==1.20.1 onnx==1.20.1 onnxruntime-gpu==1.20.1 nest-asyncio


In [ ]:
import os
from getpass import getpass

# Set your Hugging Face token only if the model requires it.
# If you already have HF_TOKEN in the environment, this cell will reuse it.
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face token (or leave blank if not needed): ")


In [ ]:
%%writefile web.py
import json
import os
from typing import Optional

import numpy as np
import torch
import torchaudio
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import HTMLResponse
from transformers import AutoModel

TARGET_SR = 16000
LANG_CODE = "bn"
DECODE_STRATEGY = "ctc"

app = FastAPI()

HF_TOKEN = os.environ.get("HF_TOKEN") or None

print("Loading ASR model...")
model = AutoModel.from_pretrained(
    "ai4bharat/indic-conformer-600m-multilingual",
    trust_remote_code=True,
    token=HF_TOKEN,
    device_map="auto",
)
model.eval()
print("ASR model ready.")

resampler_cache = {}

def get_resampler(orig_sr: int):
    if orig_sr not in resampler_cache:
        resampler_cache[orig_sr] = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=TARGET_SR)
    return resampler_cache[orig_sr]

def transcribe_int16_pcm(audio_bytes: bytes, sample_rate: int) -> str:
    if not audio_bytes:
        return ""

    audio_np = np.frombuffer(audio_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    if audio_np.size == 0:
        return ""

    wav = torch.from_numpy(audio_np).unsqueeze(0)

    if sample_rate != TARGET_SR:
        wav = get_resampler(sample_rate)(wav)

    with torch.inference_mode():
        out = model(wav, LANG_CODE, DECODE_STRATEGY)

    if isinstance(out, (list, tuple)):
        out = out[0]

    return str(out).strip()

HTML = """
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Bengali ASR Demo</title>
  <style>
    body {
      font-family: Arial, sans-serif;
      margin: 0;
      padding: 24px;
      background: #f5f7fb;
      color: #172033;
    }
    .container {
      display: grid;
      grid-template-columns: 360px 1fr;
      gap: 24px;
      align-items: stretch;
      min-height: 80vh;
    }
    .panel, .textbox {
      background: white;
      border-radius: 18px;
      box-shadow: 0 8px 30px rgba(0,0,0,0.08);
      padding: 24px;
    }
    h1 { margin-top: 0; font-size: 28px; }
    .hint { color: #52607a; line-height: 1.55; }
    #holdBtn {
      width: 100%;
      min-height: 160px;
      border: none;
      border-radius: 22px;
      font-size: 28px;
      font-weight: 700;
      cursor: pointer;
      background: #2563eb;
      color: white;
      transition: transform 0.08s ease, background 0.2s ease;
      user-select: none;
      touch-action: none;
    }
    #holdBtn.active {
      background: #dc2626;
      transform: scale(0.98);
    }
    #status {
      margin-top: 18px;
      padding: 14px 16px;
      border-radius: 14px;
      background: #eef2ff;
      min-height: 24px;
      font-size: 16px;
    }
    textarea {
      width: 100%;
      height: 72vh;
      border: 2px solid #d8e0ef;
      border-radius: 18px;
      padding: 18px;
      font-size: 28px;
      line-height: 1.5;
      resize: vertical;
      box-sizing: border-box;
    }
    .row {
      display: flex;
      gap: 12px;
      margin-top: 14px;
    }
    button.secondary {
      flex: 1;
      border: none;
      border-radius: 12px;
      padding: 14px;
      font-size: 16px;
      cursor: pointer;
      background: #e2e8f0;
    }
  </style>
</head>
<body>
  <div class="container">
    <div class="panel">
      <h1>Press and hold to speak</h1>
      <p class="hint">
        Hold the big button, speak in Bengali, and release when done.
        The transcript will keep updating on the right while you talk.
      </p>
      <button id="holdBtn">🎙️ Hold to talk</button>
      <div id="status">Connecting...</div>
      <div class="row">
        <button class="secondary" id="clearBtn">Clear text</button>
      </div>
    </div>
    <div class="textbox">
      <textarea id="transcript" placeholder="বাংলা কথাগুলো এখানে দেখা যাবে..."></textarea>
    </div>
  </div>

<script>
  const holdBtn = document.getElementById("holdBtn");
  const statusEl = document.getElementById("status");
  const transcriptEl = document.getElementById("transcript");
  const clearBtn = document.getElementById("clearBtn");

  let ws = null;
  let audioContext = null;
  let mediaStream = null;
  let sourceNode = null;
  let processorNode = null;
  let isRecording = false;
  let sampleRate = 48000;

  function setStatus(msg) {
    statusEl.textContent = msg;
  }

  function ensureWebSocket() {
    return new Promise((resolve, reject) => {
      if (ws && ws.readyState === WebSocket.OPEN) {
        resolve(ws);
        return;
      }
      const protocol = window.location.protocol === "https:" ? "wss:" : "ws:";
      ws = new WebSocket(`${protocol}//${window.location.host}/ws`);
      ws.binaryType = "arraybuffer";

      ws.onopen = () => {
        setStatus("Ready.");
        resolve(ws);
      };

      ws.onmessage = (event) => {
        try {
          const msg = JSON.parse(event.data);
          if (msg.text !== undefined) {
            transcriptEl.value = msg.text;
          }
          if (msg.status) {
            setStatus(msg.status);
          }
          if (msg.error) {
            setStatus("Error: " + msg.error);
          }
        } catch (e) {
          console.error(e);
        }
      };

      ws.onclose = () => {
        setStatus("Socket closed. Press again to reconnect.");
      };

      ws.onerror = (err) => {
        console.error(err);
        setStatus("WebSocket error.");
        reject(err);
      };
    });
  }

  function floatTo16BitPCM(float32Array) {
    const out = new Int16Array(float32Array.length);
    for (let i = 0; i < float32Array.length; i++) {
      const s = Math.max(-1, Math.min(1, float32Array[i]));
      out[i] = s < 0 ? s * 0x8000 : s * 0x7fff;
    }
    return out;
  }

  async function startRecording() {
    if (isRecording) return;

    await ensureWebSocket();

    transcriptEl.value = "";
    mediaStream = await navigator.mediaDevices.getUserMedia({
      audio: {
        channelCount: 1,
        echoCancellation: true,
        noiseSuppression: true,
        autoGainControl: true
      }
    });

    audioContext = new (window.AudioContext || window.webkitAudioContext)();
    sampleRate = audioContext.sampleRate;
    sourceNode = audioContext.createMediaStreamSource(mediaStream);
    processorNode = audioContext.createScriptProcessor(4096, 1, 1);

    ws.send(JSON.stringify({event: "start", sample_rate: sampleRate}));

    processorNode.onaudioprocess = (event) => {
      if (!isRecording || !ws || ws.readyState !== WebSocket.OPEN) return;
      const input = event.inputBuffer.getChannelData(0);
      const pcm16 = floatTo16BitPCM(input);
      ws.send(pcm16.buffer);
    };

    sourceNode.connect(processorNode);
    processorNode.connect(audioContext.destination);

    isRecording = true;
    holdBtn.classList.add("active");
    holdBtn.textContent = "🛑 Release to stop";
    setStatus("Listening...");
  }

  async function stopRecording() {
    if (!isRecording) return;
    isRecording = false;

    if (ws && ws.readyState === WebSocket.OPEN) {
      ws.send(JSON.stringify({event: "stop"}));
    }

    if (processorNode) {
      processorNode.disconnect();
      processorNode.onaudioprocess = null;
      processorNode = null;
    }
    if (sourceNode) {
      sourceNode.disconnect();
      sourceNode = null;
    }
    if (mediaStream) {
      mediaStream.getTracks().forEach(track => track.stop());
      mediaStream = null;
    }
    if (audioContext) {
      await audioContext.close();
      audioContext = null;
    }

    holdBtn.classList.remove("active");
    holdBtn.textContent = "🎙️ Hold to talk";
    setStatus("Processing final text...");
  }

  holdBtn.addEventListener("pointerdown", async (e) => {
    e.preventDefault();
    try {
      await startRecording();
    } catch (err) {
      console.error(err);
      setStatus("Could not start microphone. Make sure mic permission is allowed.");
    }
  });

  ["pointerup", "pointerleave", "pointercancel"].forEach(evt => {
    holdBtn.addEventListener(evt, (e) => {
      e.preventDefault();
      stopRecording();
    });
  });

  clearBtn.addEventListener("click", () => {
    transcriptEl.value = "";
  });

  ensureWebSocket();
</script>
</body>
</html>
"""

@app.get("/")
async def root():
    return HTMLResponse(HTML)

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()

    sample_rate: Optional[int] = None
    all_audio = bytearray()
    pending_audio = bytearray()
    bytes_per_second = None
    update_every_seconds = 2.0

    await websocket.send_json({"status": "Connected. Hold the button and start speaking in Bengali."})

    try:
        while True:
            message = await websocket.receive()

            if "text" in message and message["text"] is not None:
                payload = json.loads(message["text"])
                event = payload.get("event")

                if event == "start":
                    sample_rate = int(payload["sample_rate"])
                    bytes_per_second = sample_rate * 2
                    all_audio = bytearray()
                    pending_audio = bytearray()
                    await websocket.send_json({"status": "Mic started. Speak now..."})

                elif event == "stop":
                    if sample_rate and len(all_audio) > 0:
                        transcript = transcribe_int16_pcm(bytes(all_audio), sample_rate)
                        await websocket.send_json({"text": transcript, "status": "Done."})
                    else:
                        await websocket.send_json({"status": "No audio captured."})

            if "bytes" in message and message["bytes"] is not None:
                chunk = message["bytes"]
                all_audio.extend(chunk)
                pending_audio.extend(chunk)

                if sample_rate and bytes_per_second and len(pending_audio) >= int(bytes_per_second * update_every_seconds):
                    transcript = transcribe_int16_pcm(bytes(all_audio), sample_rate)
                    await websocket.send_json({"text": transcript, "status": "Transcribing..."})
                    pending_audio = bytearray()

    except WebSocketDisconnect:
        print("Client disconnected")
    except Exception as exc:
        await websocket.send_json({"error": str(exc)})


In [ ]:
import subprocess
import time
import signal

# Stop an earlier server process if you already started one in this runtime.
try:
    server.terminate()
    time.sleep(2)
except:
    pass

server = subprocess.Popen(
    ["uvicorn", "web:app", "--host", "0.0.0.0", "--port", "8000"],
)
time.sleep(8)
print("Uvicorn started on port 8000")


In [ ]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok
from IPython.display import HTML, display

# Replace this with your own ngrok auth token if needed
if "NGROK_AUTHTOKEN" in os.environ:
    ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

try:
    ngrok.kill()
except:
    pass

public_url = ngrok.connect(8000, bind_tls=True).public_url
print("Open this URL in a new browser tab:")
print(public_url)

display(HTML(f'''
<div style="padding:16px;border:1px solid #ddd;border-radius:12px;background:#fafafa">
  <h3>ASR app is ready</h3>
  <p><a href="{public_url}" target="_blank">{public_url}</a></p>
  <p>Press and hold the microphone button, speak in Bengali, then release.</p>
</div>
'''))


In [ ]:
# Optional: stop the server and tunnel when finished.
# server.terminate()
# ngrok.kill()
